# Import libraries

In [1]:
import numpy as np
import pandas as pd
import os
import uuid
from datetime import datetime
from tqdm import tqdm
import torch
from transformers import (
    RobertaForSequenceClassification,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig
)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

import psycopg2
from dotenv import load_dotenv
from sqlalchemy import create_engine, Table, MetaData, update, insert, select, bindparam, and_
load_dotenv()

True

# Connect to database


## Get database information


In [4]:
postgres_user = os.getenv("POSTGRES_USER")
postgres_password = os.getenv("POSTGRES_PASSWORD")
postgres_host = os.getenv("POSTGRES_HOST")
postgres_port = os.getenv("POSTGRES_PORT")
postgres_db = os.getenv("POSTGRES_DB")

In [5]:
try:
    conn = psycopg2.connect(
        database=postgres_db,
        user=postgres_user,
        host=postgres_host,
        password=postgres_password,
        port=postgres_port,
    )
    print("Opened database successfully")
except Exception as e:
    print(f"Connection failed: {e}")

Opened database successfully


In [6]:
engine = create_engine(
    f"postgresql://{postgres_user}:{postgres_password}@{postgres_host}:{postgres_port}/{postgres_db}"
)

# Import PhoBert and VisoBert

In [8]:
model_path = "wonrax/phobert-base-vietnamese-sentiment"
wonrax = RobertaForSequenceClassification.from_pretrained(model_path)
wonrax_tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)

In [9]:
model_path = '5CD-AI/Vietnamese-Sentiment-visobert'
uit_tokenizer = AutoTokenizer.from_pretrained(model_path)
uit_config = AutoConfig.from_pretrained(model_path)
uit = AutoModelForSequenceClassification.from_pretrained(model_path)

# Get news from database

In [7]:
# Define metadata and table object
metadata = MetaData()
news_table = Table('news', metadata, autoload_with=engine)

# Define the query
news_query = select(news_table.c.title, news_table.c.rawContent, news_table.c.cleanedContent, news_table.c.company, news_table.c.url, news_table.c.id)

# Execute the query and fetch the data
news_info = pd.read_sql(news_query, engine)

# Create the dictionary
news_dict = {
    "title": news_info["title"].tolist(),
    "rawContent": news_info["rawContent"].tolist(),
    "cleanedContent": news_info["cleanedContent"].tolist(),
    "company": news_info["company"].tolist(),
    "url": news_info["url"].tolist(),
    "id": news_info["id"].tolist()
}

In [8]:
news = pd.DataFrame(news_dict)
news.notnull().sum()

title             72878
rawContent        72878
cleanedContent    72878
company           72878
url               72878
id                72878
dtype: int64

# Generate UUID

In [7]:
def generate_uuid(model):
    id = str(uuid.uuid4())
    created_at = datetime.now()
    updated_at = created_at
    return id, created_at, updated_at

# Analyze data

In [22]:
company = "FPT"

company_table = Table('company', metadata, autoload_with=engine)
company_query = select(company_table.c.id).where(company_table.c.symbol == company)
company_id = pd.read_sql(company_query, engine)["id"][0]
company_id

news_table = Table('news', metadata, autoload_with=engine)
from_date = datetime(2024, 5, 1)  # May 1, 2024
to_date = datetime(2024, 7, 31)   # July 31, 2024

# Query the news table for the specified company_id and date range
news_query = select(news_table.c.id, news_table.c.cleanedContent, news_table.c.publishedAt).where(
    and_(
        news_table.c.company == company_id,
        news_table.c.publishedAt >= from_date,
        news_table.c.publishedAt <= to_date
    )
)

# Execute the query and fetch the results
news_df = pd.read_sql(news_query, engine)
news_id = news_df["id"].tolist()
news_date = {
    news_df["id"][i]: news_df["publishedAt"][i] for i in range(len(news_df))
}

# Query the sentiment table for the specified news_id
sentiment_table = Table('sentiment', metadata, autoload_with=engine)
sentiment_query = select(sentiment_table.c.score, sentiment_table.c.modelName, sentiment_table.c.news).where(sentiment_table.c.news.in_(news_id))
sentiment_df = pd.read_sql(sentiment_query, engine)

phobert_df = sentiment_df[sentiment_df["modelName"] == "PhoBert"]
phobert_df.reset_index(drop=True, inplace=True)

In [19]:
phobert_df.notnull().sum()

score        75
modelName    75
news         75
dtype: int64

In [26]:
# sort news date by date
news_date = {k: v for k, v in sorted(news_date.items(), key=lambda item: item[1])}
news_date

{'8d7ac20c-9aab-40b0-a4e4-fb73b2eb3ec7': Timestamp('2024-05-02 00:00:00'),
 '5a6becdd-c7a7-446f-a5c9-913c2200290e': Timestamp('2024-05-03 00:00:00'),
 '8efc20fa-6c55-4e6d-9f68-01c4e6a39ec2': Timestamp('2024-05-06 00:00:00'),
 'd90160db-56c6-4bdf-a264-f6d4340ea413': Timestamp('2024-05-08 00:00:00'),
 'b947f411-36f9-458f-a141-ebace1f2b2c2': Timestamp('2024-05-08 00:00:00'),
 '63ee8aa3-af86-49d8-b687-50a2f1ad1fda': Timestamp('2024-05-09 00:00:00'),
 'b9570a93-aeca-47b2-9409-41f0585f4f30': Timestamp('2024-05-10 00:00:00'),
 'bdacafdc-d32c-42d2-941e-0f1591b7b7ca': Timestamp('2024-05-14 00:00:00'),
 'c730febc-6974-4603-b016-dc479ae2c6ea': Timestamp('2024-05-16 00:00:00'),
 '8595b5a6-7c8e-4fc3-9b04-95e314761d88': Timestamp('2024-05-16 00:00:00'),
 '80b0841f-a386-42cd-b9c8-e2b1ff94f029': Timestamp('2024-05-20 00:00:00'),
 '4675341d-346b-4f3c-ac00-776c1debdaee': Timestamp('2024-05-21 00:00:00'),
 'd24ba88c-9c0d-4084-be61-2332973ac32e': Timestamp('2024-05-21 00:00:00'),
 'b615af20-1cce-448b-9215

In [23]:
print(phobert_df)

    score modelName                                  news
0   0.991   PhoBert  7ad9f8ba-8d5e-4f1b-8b8f-1fcb1b9271d0
1   0.963   PhoBert  bde1640a-03ed-4478-b1dd-7e6df2cf2f69
2   0.100   PhoBert  f1d9f246-d59f-4722-93a7-12be4b9881e5
3   0.025   PhoBert  2107949f-507f-43d5-a174-14188dbc24fb
4   0.989   PhoBert  2db8aaa7-7061-4da7-ba03-139c9f54a28a
..    ...       ...                                   ...
70 -0.188   PhoBert  b615af20-1cce-448b-9215-7adb7b65f3aa
71  0.932   PhoBert  4675341d-346b-4f3c-ac00-776c1debdaee
72  0.966   PhoBert  d24ba88c-9c0d-4084-be61-2332973ac32e
73  0.945   PhoBert  80b0841f-a386-42cd-b9c8-e2b1ff94f029
74  0.958   PhoBert  8595b5a6-7c8e-4fc3-9b04-95e314761d88

[75 rows x 3 columns]


In [31]:
for news in news_date.keys():
    news_id = news
    score = phobert_df[phobert_df["news"] == news_id]["score"].values[0]
    print(news_date[news_id], score)

2024-05-02 00:00:00 0.983
2024-05-03 00:00:00 0.973
2024-05-06 00:00:00 0.069
2024-05-08 00:00:00 0.952
2024-05-08 00:00:00 0.968
2024-05-09 00:00:00 0.984
2024-05-10 00:00:00 0.986
2024-05-14 00:00:00 0.903
2024-05-16 00:00:00 0.606
2024-05-16 00:00:00 0.958
2024-05-20 00:00:00 0.945
2024-05-21 00:00:00 0.932
2024-05-21 00:00:00 0.966
2024-05-22 00:00:00 -0.188
2024-05-23 00:00:00 0.156
2024-05-23 00:00:00 0.841
2024-05-24 00:00:00 0.301
2024-05-24 00:00:00 0.964
2024-05-27 00:00:00 0.988
2024-05-29 00:00:00 0.989
2024-05-30 00:00:00 -0.143
2024-05-30 00:00:00 0.745
2024-05-31 00:00:00 0.735
2024-05-31 00:00:00 0.985
2024-06-03 00:00:00 0.848
2024-06-04 00:00:00 0.815
2024-06-04 00:00:00 -0.39
2024-06-04 00:00:00 0.985
2024-06-05 00:00:00 0.934
2024-06-05 00:00:00 0.164
2024-06-06 00:00:00 0.007
2024-06-06 00:00:00 0.975
2024-06-06 00:00:00 0.961
2024-06-07 00:00:00 -0.13
2024-06-09 00:00:00 0.056
2024-06-09 00:00:00 0.79
2024-06-10 00:00:00 0.928
2024-06-10 00:00:00 0.983
2024-06-11 

In [40]:
sentiment_news_query = "SELECT news FROM sentiment"
sentiment_info = pd.read_sql(sentiment_news_query, engine)

# Get duplicated news
duplicated_news = sentiment_info[sentiment_info.duplicated()]

# Get count of each duplicated news item
duplicated_news_count = sentiment_info[sentiment_info.duplicated(keep=False)]['news'].value_counts()

# Get the duplicated news ids if count == 4
duplicated_news_ids = duplicated_news_count[duplicated_news_count == 4].index.tolist()
duplicated_news_ids

['5447416d-bffe-46db-94a4-fca60ebe5238']

In [16]:
# remove news_ids that have been in sentiment_news
news_ids = [news_id for news_id in news_ids if news_id not in sentiment_news]
len(news_ids)

0

In [14]:
sentiment_table = Table('sentiment', metadata, autoload_with=engine)

insert_sentiment = insert(sentiment_table).values(
    id=bindparam('b_id'),
    modelName=bindparam('b_modelName'),
    score=bindparam('b_score'),
    order=bindparam('b_order'),
    news=bindparam('b_news'),
    createdAt=bindparam('b_created_at'),
    updatedAt=bindparam('b_updated_at')
)

select_sentiment = select(sentiment_table.c.id).where(
    (sentiment_table.c.news == bindparam('b_news')) & 
    (sentiment_table.c.modelName == bindparam('b_modelName'))
)

data = []
count = 0
batch_size = 1000
max_length = 256  # Define the maximum sequence length

for i in tqdm(range(len(news_dict["cleanedContent"]))):
    content = news_dict["cleanedContent"][i]
    if pd.isnull(content) or pd.isna(content) or len(content) == 0:
        continue
    content_id = news_dict["id"][i]

    if content_id in sentiment_news:
        continue

    wonrax_sentiment = dict()
    try: 
        input_ids = torch.tensor([wonrax_tokenizer.encode(content, max_length=max_length, truncation=True)])
        with torch.no_grad():
            out = wonrax(input_ids)
            wonrax_sentiment["neg"] = round(float(out.logits.softmax(dim=-1).tolist()[-1][0]), 3)
            wonrax_sentiment["pos"] = round(float(out.logits.softmax(dim=-1).tolist()[-1][1]), 3)
            wonrax_sentiment["neu"] = round(float(out.logits.softmax(dim=-1).tolist()[-1][2]), 3)
            wonrax_sentiment["compound"] = round(
                wonrax_sentiment["pos"] - wonrax_sentiment["neg"], 4
            )
    except:
        print(f"Error at {content_id}, {len(content)} using PhoBERT")
        wonrax_sentiment["compound"] = 0

    sentiment_id, created_at, updated_at = generate_uuid("sentiment_phobert")

    data.append({"b_id": sentiment_id, "b_modelName": "PhoBert", "b_score": wonrax_sentiment["compound"], "b_order": 0, "b_news": content_id, "b_created_at": created_at, "b_updated_at": updated_at})

    uit_sentiment = dict()

    try:
        input_ids = torch.tensor([uit_tokenizer.encode(content.replace('_', ' '), max_length=max_length, truncation=True)])
        with torch.no_grad():
            output = uit(input_ids)
            scores = output.logits.softmax(dim=-1).cpu().numpy()[0]
            ranking = np.argsort(scores)
            ranking = ranking[::-1]
            for i in range(scores.shape[0]):
                label = uit_config.id2label[ranking[i]]
                score = float(scores[ranking[i]])  # Convert numpy.float32 to native float
                uit_sentiment[label.lower()] = score
            uit_sentiment['compound'] = round(uit_sentiment['pos'] - uit_sentiment['neg'], 4)
    except:
        print(f"Error at {content_id}, {len(content)} using ViSoBert")
        uit_sentiment['compound'] = 0

    sentiment_id, created_at, updated_at = generate_uuid("sentiment_visobert")

    data.append({"b_id": sentiment_id, "b_modelName": "VisoBert", "b_score": uit_sentiment["compound"], "b_order": 0, "b_news": content_id, "b_created_at": created_at, "b_updated_at": updated_at})

    count += 1

    if count % batch_size == 0:
        print("Inserting data") 
        with engine.connect() as conn:
            conn.execute(insert_sentiment, data)
            conn.commit()
            data = []

if data:
    with engine.connect() as conn:
        conn.execute(insert_sentiment, data)
        conn.commit()  # Ensure changes are committed   

  5%|▍         | 3522/72878 [05:38<8:58:52,  2.15it/s]

Inserting data


  6%|▋         | 4652/72878 [11:37<6:25:44,  2.95it/s] 

Inserting data


  8%|▊         | 5792/72878 [17:29<7:32:28,  2.47it/s] 

...

 96%|█████████▌| 69977/72878 [4:45:07<13:39,  3.54it/s]

Inserting data


 98%|█████████▊| 71408/72878 [4:50:46<05:07,  4.77it/s]

Inserting data


100%|██████████| 72878/72878 [4:54:37<00:00,  4.12it/s]


In [8]:
news = pd.DataFrame(news_dict)

# Get top 10 company with the most news
news["company"].value_counts().head(10)

# Get top 10 company with the least news
news["company"].value_counts().tail(10)

company
f740f020-a989-45e6-a019-ae78b5ca304f    1
6ac99c69-3ce9-4b22-be3a-603935c3e14e    1
40c762cf-0b7a-4f89-8ee5-62e293806a2c    1
3b71f5f4-19da-4d5d-894a-a7602974d953    1
8e71a9f9-8f08-4b8d-ac37-d959ae90bcee    1
64117afc-f706-4ede-a538-64bc6988f3d2    1
7d050a3c-59b7-4d37-ad1b-2d6f2011262e    1
a9069a4d-435e-4ca7-8c11-821b314407f6    1
336fba1a-3906-41fa-b5a9-5dbab2532ca5    1
e8971a6e-3b14-4fc6-a866-cb371f535543    1
Name: count, dtype: int64